# 第9讲：SVM怎样确定边界？

配合 `student-worksheet.md`，先写预测，再运行。数据是UCI Vertebral Column真实记录，
不是模拟患者。两个输入预先固定，八点子集仅用于几何，正式实验使用完整训练集。

本Notebook不自动保存图表和结果。需要导出时只写项目根目录 `.tmp/svm/`。
默认不读取测试标签。G为可选扩展，放在最后测试之前，避免根据测试追加解释或调参。


In [1]:
import os, sys
from pathlib import Path
sys.dont_write_bytecode = True
here = Path.cwd().resolve()
lesson = next((q for p in [here, *here.parents]
               for q in [p, p/'class2026/materials/9. SVM']
               if (q/'scripts/lab_utils.py').is_file()), None)
if lesson is None:
    raise RuntimeError('请从项目根目录、本课文件夹或本课Notebook所在目录运行。')
repo = lesson.parents[2]
out = repo/'.tmp/svm/student'
out.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str(repo/'.tmp/svm/matplotlib'))
sys.path.insert(0, str(lesson/'scripts'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from lab_utils import *
pd.set_option('display.max_columns', 12)


## A．记录、输入和目标

先完成工作单A。数据说明见 `data/README.md`。
图先显示全部训练范围，极端值也保留。后续边界图放大主要区域时会报告范围外点数，
范围外的点仍参与拟合。不要因为极值影响画图就删除记录。


In [2]:
train, validation = load_development()
display(train.head())
display(pd.DataFrame({'train': train.abnormal.value_counts(),
                      'validation': validation.abnormal.value_counts()}).sort_index())
assert set(train.sample_id).isdisjoint(validation.sample_id)
fig, ax = plt.subplots(figsize=(8, 5))
for cls, color, marker in [(0, '#087c77', 'o'), (1, '#b75d24', '^')]:
    data = train[train.abnormal == cls]
    ax.scatter(data[FEATURES[0]], data[FEATURES[1]], color=color,
               marker=marker, alpha=.7, label=['NO', 'AB'][cls])
ax.set(xlabel='Pelvic incidence (source scale)', ylabel='Spondylolisthesis (source scale)',
       title='All 186 training records: full extent')
ax.legend(); plt.show()


,sample_id,pelvic_incidence,degree_spondylolisthesis,abnormal
0,V135,52.20,54.94,1
1,V150,79.25,36.71,1
2,V040,52.42,1.69,1
3,V257,54.75,8.24,0
4,V294,48.90,19.85,0


,train,validation
abnormal,,
0,60,20
1,126,42


C:\Users\Murmure\AppData\Local\Temp\ipykernel_32600\3711503345.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.legend(); plt.show()


## B．三条候选线与距离

先观察图，不运行下一格数值核对。三条线的法向量相同，截距不同。
按工作单手算V061的分数与距离，解释同乘3后发生什么，再计算每条线最近的带标签距离。

`geometry_experiment`从训练集有意选择4个NO与4个AB；scaler拟合全训练集。
其大C拟合经核对近似展示可分数据的hard margin，不是专用hard-margin求解器。


### B 手算（先于运行完成）

`f = w·z + b = .7287×.7108 + .7492×.6700 − .0200 ≈ 1.0000` → f>0 → **预测 AB**。
`||w|| = √(.7287²+.7492²) ≈ 1.0451`，`距离 = |f|/||w|| ≈ .9568`。
→ 与下一格 `score=1.000000, distance=.956807` 一致。**f 不是百分比风险**，只是带符号决策分数。

w、b 同乘 3：边界与预测不变，f→3f、||w||→3||w||，**距离不变**。
因此比较不同边界只看 |f| 不够，必须用 |f|/||w|| 归一化。

In [3]:
geo = geometry_experiment(train)
z, y = geo['z'], geo['subset'].abnormal.to_numpy()
w, b = geo['model'].coef_[0], geo['model'].intercept_[0]
fig, ax = plt.subplots(figsize=(7, 6))
for cls, color, marker in [(0, '#087c77', 'o'), (1, '#b75d24', '^')]:
    mask = y == cls
    ax.scatter(z[mask, 0], z[mask, 1], c=color, marker=marker, label=['NO', 'AB'][cls])
for i, name in enumerate(geo['subset'].sample_id):
    ax.annotate(name, z[i], xytext=(5, 5), textcoords='offset points')
xx = np.linspace(-2.1, 1.8, 100)
for offset, color in [(-.6, '#9c562b'), (0, '#193638'), (.6, '#5c7b9c')]:
    ax.plot(xx, -(w[0]*xx+b+offset)/w[1], color=color, label=f'b offset {offset:+.1f}')
ax.set(xlabel='z1', ylabel='z2', xlim=(-2.1, 1.8), ylim=(-1.3, 2.2), aspect='equal')
ax.legend(); plt.show()
display(geo['table'][['sample_id', 'z1', 'z2', 'abnormal']])


C:\Users\Murmure\AppData\Local\Temp\ipykernel_32600\407668481.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.legend(); plt.show()


,sample_id,z1,z2,abnormal
0,V213,-0.995691,-0.556635,0
1,V215,-0.919516,-0.757645,0
2,V216,-1.769943,-0.724543,0
3,V217,-0.623345,-0.701755,0
4,V061,0.710846,0.670049,1
5,V068,0.783042,0.991952,1
6,V072,1.422567,1.763607,1
7,V073,1.312853,1.106609,1


先把你的预测与手算写在工作单B，之后运行下面的核对。


### B 三条候选截距的核对结果

| b偏移 | 训练错误 | 最小带符号距离 |
|---|---|---|
| −.6 | 0 | .382723 |
| 0 | 0 | .956807 |
| +.6 | 0 | .382723 |

三条线训练错误都是 0，但最近点距离不同 → **零错误不能选唯一边界，最大间隔才能**。
带宽 `2/||w|| ≈ 1.913613`（标准化空间）；单边最小距离 `1/||w|| ≈ .956807`，相差**因子 2**。

In [4]:
print('w =', w, 'b =', b, '||w|| =', np.linalg.norm(w))
display(geo['candidates'])
display(geo['table'][['sample_id', 'score', 'signed_margin', 'distance', 'support']])
print('Full canonical band width:', 2/np.linalg.norm(w))
draw_boundary(geo['model'], train, 'Geometry subset: f = -1, 0, +1',
              True, geo['scaler'], geo['subset'])
plt.show()


w = [0.72868524 0.74922795] b = -0.020002722800898765 ||w|| = 1.0451433912504728


,offset,b,training_errors,min_signed_distance
0,-0.6,-0.620003,0,0.382723
1,0.0,-0.020003,0,0.956807
2,0.6,0.579997,0,0.382723


,sample_id,score,signed_margin,distance,support
0,V213,-1.162595,1.162595,1.112378,False
1,V215,-1.257689,1.257689,1.203365,False
2,V216,-1.852582,1.852582,1.772562,False
3,V217,-1.000000,1.000000,0.956807,True
4,V061,1.000000,1.000000,0.956807,True
5,V068,1.293786,1.293786,1.237903,False
6,V072,2.337944,2.337944,2.236960,False
7,V073,1.765756,1.765756,1.689487,False


Full canonical band width: 1.9136130187907316


C:\Users\Murmure\AppData\Local\Temp\ipykernel_32600\2296177359.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## C．支持向量与矛盾标签

先预测哪种删除会改变边界，以及重估scaler会混入什么因素。
本格比较删除非SV V072与SV V061，scaler冻结。

矛盾实验另外复制一点、给相反标签，只是人工反事实，不修改任何原始数据。
先写出hard-margin约束的矛盾，再看soft-margin拟合。不能把结果解释为原数据有错标。


### C 删除实验与 hinge 手算

| 删除 | ID | 八点分数最大变化 | 解释 |
|---|---|---|---|
| 非SV | V072 | 0.000000 | Δw=Δb=0，边界不变 |
| SV | V061 | 0.427454 | 约束点移走，边界旋转/平移 |

SV 只是当前特征+参数+训练集下的约束点，**不是最典型/最严重的患者**。
同一坐标赋相反标签：需同时满足 `f≥1` 与 `f≤−1` → `1≤−1` 无解；**kernel 也解决不了**（确定性函数一个输入只有一个符号）。
soft margin 允许违例后目标仍有有限最小值：C=.01/1/100 → 错误 4/1/1，总 hinge 7.287593/2/2。零训练错误既不一定可能也不一定合理。

hinge 手算：y·f = −.5 / .4 / 1 / 1.8 → hinge = **1.5 / .6 / 0 / 0**；分错 / 分对但进带 / 边界上 / 带外。

In [5]:
display(geo['changes'])
display(geo['contradiction'])
yf = np.array([-.5, .4, 1., 1.8])
display(pd.DataFrame({'y_times_f': yf, 'hinge': np.maximum(0, 1-yf), 'misclassified': yf < 0}))


,operation,sample_id,coefficient_change,intercept_change,score_change_max
0,delete_non_support,V072,0.000000,0.000000,0.000000
1,delete_support,V061,0.156617,0.127752,0.427454


,C,errors,sum_hinge,n_sv
0,0.01,4,7.287593,8
1,1.00,1,2.000000,4
2,100.00,1,2.000000,4


,y_times_f,hinge,misclassified
0,-0.5,1.5,True
1,0.4,0.6,False
2,1.0,0.0,False
3,1.8,0.0,False


## D．只改变linear C

先用目标 `½||w||²+CΣhinge` 预测改变方向。下面统一拟合全部预定配置，后续重复使用。
先看linear切片，别先寻找最高分。mean_hinge乘186才是目标中的hinge总和。
线性宽度在训练标准化空间中，soft margin允许参考带内有点。


### D 预测与解读

预测：C 增大 → 违例更贵 → 总 hinge 不增、宽度 2/||w|| 不增；但**验证表现不必同步变化**。
实测：C=.01→100，宽度 12.413632→.489468，平均 hinge .631206→.398692，训练错误 60→35，SV 121→76，验证 **.500000→.759524→.709524（非单调）**（最优 C=1）。
- 训练错误≠hinge：后者还惩罚 0<yf<1 的点。
- 不能跨 C 比较目标值（C 改了目标本身，7424.01 与 80.93 不同标尺）。
- C=.01 宽度最大却验证 .500000 → **只看"宽度大"不够**。
- f=±1 之间允许有点；宽 margin 不保证未来表现。

In [6]:
grid, models = parameter_experiment(train, validation)
linear = grid[grid.kernel == 'linear']
display(linear[['C', 'linear_width', 'mean_hinge', 'regularizer', 'objective',
                'train_errors', 'n_sv', 'val_balanced_accuracy']])
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for ax, column in zip(axes, ['linear_width', 'mean_hinge', 'val_balanced_accuracy']):
    ax.semilogx(linear.C, linear[column], 'o-', color='#087c77')
    ax.set(xlabel='C', ylabel=column)
fig.tight_layout(); plt.show()
for C in [.01, 1., 100.]:
    draw_boundary(models[('linear', C, .1)], train, f'Linear C={C}')
    plt.show()


C:\Users\Murmure\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Users\Murmure\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Users\Murmure\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Users\Murmure\AppData\Local\Programs\Python\Python312\Lib\site-packages\skle

C:\Users\Murmure\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Users\Murmure\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


C:\Users\Murmure\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Users\Murmure\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Users\Murmure\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Users\Murmure\AppData\Local\Programs\Python\Python312\Lib\site-packages\skle

C:\Users\Murmure\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Users\Murmure\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


C:\Users\Murmure\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Users\Murmure\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Users\Murmure\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Users\Murmure\AppData\Local\Programs\Python\Python312\Lib\site-packages\skle

,C,linear_width,mean_hinge,regularizer,objective,train_errors,n_sv,val_balanced_accuracy
0,0.01,12.413632,0.631206,0.012979,1.187021,60,121,0.500000
1,0.10,1.241363,0.505605,1.297873,10.702127,40,121,0.734524
2,1.00,0.605240,0.405765,5.459779,80.932033,37,89,0.759524
3,10.00,0.491488,0.398717,8.279506,749.893444,36,77,0.709524
4,100.00,0.489468,0.398692,8.347989,7424.010939,35,76,0.709524


C:\Users\Murmure\AppData\Local\Temp\ipykernel_32600\3976442274.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); plt.show()
C:\Users\Murmure\AppData\Local\Temp\ipykernel_32600\3976442274.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\Murmure\AppData\Local\Temp\ipykernel_32600\3976442274.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\Murmure\AppData\Local\Temp\ipykernel_32600\3976442274.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## E．kernel与gamma的单因素切片

先手算距离1/2下两种gamma的RBF相似度。固定C=1看gamma，再固定gamma=10看C。
最后才读联合网格。原平面RBF的f=±1是分数轮廓，不能当等距margin线。
Polynomial仅展示一个配置，不展开搜索。


### E 手算与解读

| d | gamma=.1 | gamma=1 |
|---|---|---|
| 1 | .904837 | .367879 |
| 2 | .670320 | .018316 |

gamma 越大 → 衰减越快 → 影响更**局部**。
固定 C=1：gamma .01/.1/1/10 → 训练 .500000/.848016/.864286/.870238，验证 .500000/.772619/.784524/.769048。
固定 gamma=10：C 1/10/100 → 训练 .870238/.870238/.910317，验证 .769048/.804762/**.669048**（训练更贴合，验证不持续提高）。
C 与 gamma 有交互 → 必须单因素切片，不能把变化同时归给两者。
RBF 的 f=±1 是**等值轮廓线**，不是原平面等距线，不能与 linear 的 2/||w|| 比较。

In [7]:
for gamma in [.1, 1.]:
    print('gamma', gamma, 'K at distance 1, 2:', np.exp(-gamma*np.array([1., 2.])**2))
rbf = grid[grid.kernel == 'rbf']
columns = ['C', 'gamma', 'train_balanced_accuracy', 'val_balanced_accuracy', 'n_sv']
display(rbf[rbf.C == 1][columns])
display(rbf[rbf.gamma == 10][columns])
for gamma in [.01, 1., 10.]:
    draw_boundary(models[('rbf', 1., gamma)], train, f'RBF C=1, gamma={gamma}')
    plt.show()
display(rbf.pivot(index='C', columns='gamma', values='val_balanced_accuracy'))
for C in [10., 100.]:
    draw_boundary(models[('rbf', C, 10.)], train, f'RBF C={C}, gamma=10')
    plt.show()
for key in [('linear', 1., .1), ('poly', 1., .1), ('rbf', 1., .1)]:
    draw_boundary(models[key], train, f'Kernel comparison: {key}')
    plt.show()


gamma 0.1 K at distance 1, 2: [0.90483742 0.67032005]
gamma 1.0 K at distance 1, 2: [0.36787944 0.01831564]


,C,gamma,train_balanced_accuracy,val_balanced_accuracy,n_sv
9,1.0,0.01,0.500000,0.500000,121
10,1.0,0.10,0.848016,0.772619,116
11,1.0,1.00,0.864286,0.784524,99
12,1.0,10.00,0.870238,0.769048,128


,C,gamma,train_balanced_accuracy,val_balanced_accuracy,n_sv
8,0.1,10.0,0.512698,0.500000,150
12,1.0,10.0,0.870238,0.769048,128
16,10.0,10.0,0.870238,0.804762,108
20,100.0,10.0,0.910317,0.669048,106


C:\Users\Murmure\AppData\Local\Temp\ipykernel_32600\2043108956.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\Murmure\AppData\Local\Temp\ipykernel_32600\2043108956.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\Murmure\AppData\Local\Temp\ipykernel_32600\2043108956.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


gamma,0.01,0.10,1.00,10.00
C,,,,
0.1,0.500000,0.500000,0.614286,0.500000
1.0,0.500000,0.772619,0.784524,0.769048
10.0,0.772619,0.759524,0.771429,0.804762
100.0,0.759524,0.746429,0.746429,0.669048


C:\Users\Murmure\AppData\Local\Temp\ipykernel_32600\2043108956.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\Murmure\AppData\Local\Temp\ipykernel_32600\2043108956.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\Murmure\AppData\Local\Temp\ipykernel_32600\2043108956.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\Murmure\AppData\Local\Temp\ipykernel_32600\2043108956.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\Murmure\AppData\Local\Temp\ipykernel_32600\2043108956.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## F1．同信息、不同单位

先预测x₁乘100会怎样改变距离。这里固定C=1、gamma=.1，不重新选择参数。
两个因素：单位是否改变、是否使用训练标准化。
只比较分数不变性，不把此设置当作各流程的最佳性能。


### F1 预测与结果

预测：**训练标准化**那一对分数几乎不变；未标准化那一对明显不同。
实测：标准化后 max|Δf| ≈ **1.8×10⁻¹⁴**；未标准化 ≈ **1.537660**（验证 .704762 → .488095）。
原因：x₁×100 使欧氏距离中 x₁ 分量放大 10000 倍，RBF 实际局部范围改变；StandardScaler 用训练 μ、σ 正好抵消正比例缩放。
若在全 310 条上拟合 scaler，μ、σ 含保留集输入分布 → **无标签也会泄漏**。

In [8]:
scaling, score_changes = scaling_experiment(train, validation)
display(scaling[['scaled', 'unit', 'val_balanced_accuracy', 'n_sv']])
print('max score change by scaled=False/True:', score_changes)


C:\Users\Murmure\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Users\Murmure\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Users\Murmure\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Users\Murmure\AppData\Local\Programs\Python\Python312\Lib\site-packages\skle

,scaled,unit,val_balanced_accuracy,n_sv
0,False,original,0.704762,171
1,False,x1_times_100,0.488095,186
2,True,original,0.772619,116
3,True,x1_times_100,0.772619,116


max score change by scaled=False/True: {'False': 1.5376598755511461, 'True': 1.7763568394002505e-14}


## G．可选扩展：冻结模型的输入扰动

先冻结两个家族的验证配置。规则是验证balanced accuracy最高，平分取较小C再较小gamma。
这不是跨家族选冠军。扩展在测试之前进行，扰动假设为独立高斯噪声，不是真实测量误差。
默认跳过。即使不运行扩展，也会生成供F2记录的配置。


### G 预测与结果

预测：离边界近（|f|≈0）的点轻微扰动就翻转，翻转率随噪声强度上升。
实测翻转率（0/5/10/20% 训练SD）：linear 0 / .030 / .054 / .121；RBF 0 / .070 / .138 / **.220**。
本配置 RBF 更敏感（边界更局部），但只是**本配置**结论。
噪声是人为独立高斯扰动，不等于真实仪器误差或人群稳定性；重复SD只是扰动波动，不是置信区间。

In [9]:
configs = select_validation_configs(grid)
print('Validation-only configurations:', configs)
RUN_OPTIONAL_NOISE = True
if RUN_OPTIONAL_NOISE:
    for family, key in configs.items():
        print(family)
        display(noise_experiment(models[key], train, validation))


Validation-only configurations: {'linear': ('linear', 1.0, 0.1), 'rbf': ('rbf', 10.0, 10.0)}
linear


,noise_sd_fraction,repeats,mean_balanced_accuracy,sd_balanced_accuracy,mean_flip_rate
0,0.00,1,0.759524,0.000000,0.000000
1,0.05,100,0.749869,0.022139,0.030000
2,0.10,100,0.752321,0.029517,0.054355
3,0.20,100,0.730024,0.047067,0.121129


rbf


,noise_sd_fraction,repeats,mean_balanced_accuracy,sd_balanced_accuracy,mean_flip_rate
0,0.00,1,0.804762,0.000000,0.000000
1,0.05,100,0.746345,0.032520,0.069677
2,0.10,100,0.719726,0.050828,0.137581
3,0.20,100,0.682869,0.056599,0.220323


## F2．记录后打开保留集

先完成工作单F的配置记录，并写出测试要检查什么。默认两个开关均为False。
函数在开关关闭时不读取测试标签。按教师安排发放标签后再开启。

训练仍为原186条，不把验证重新加入；不以测试继续搜索。
教师预跑已查看测试，此处只演示保留集流程，不宣称全新独立科研确认。
原生SVM分数不是概率，ROC-AUC用分数排序。


### F2 记录（先写问题，再看测试）

| 家族 | 输入、scaler范围 | C、kernel、gamma | 验证 balanced acc | 测试前想检查 |
|---|---|---|---|---|
| linear | 2输入；scaler 仅拟合186训练 | 1, linear, (.1) | .759524 | 线性够不够；FN 在哪侧；AUC是否>0.5 |
| RBF | 同上 | 10, rbf, 10 | .804762 | 局部团块是否过拟合；扰动稳健性(G) |

始终预测 AB：验证 accuracy = 42/62 ≈ **.677419**，sensitivity 1、specificity 0，balanced accuracy **.500000**。

锁定测试（62条）：linear TN19/FP1/FN12/TP30，balanced acc .832143，AUC .823810；
RBF TN19/FP1/FN11/TP31，balanced acc .844048，AUC .814286。
主要错误是 **FN**（AB 被判成 NO，漏检），FP 仅 1 例。
同源、62 条、单一机构 → **不能外推到新医院**；测试后不再调参。

In [10]:
FREEZE_CONFIRMED = True
OPEN_TEST = True
display(evaluate_locked(configs, freeze_confirmed=FREEZE_CONFIRMED, open_test=OPEN_TEST))


C:\Users\Murmure\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Users\Murmure\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


,family,C,gamma,n,TN,FP,...,TP,accuracy,balanced_accuracy,sensitivity,specificity,roc_auc
0,linear,1.0,0.1,62,19,1,...,30,0.790323,0.832143,0.714286,0.95,0.823810
1,rbf,10.0,10.0,62,19,1,...,31,0.806452,0.844048,0.738095,0.95,0.814286


## 解释与提交

完成工作单的一页概念解释：至少用两个受控实验说明机制、一个优势、一个局限和适用条件。
保留你在运行前的预测。不要提交整份自动输出，也不要用测试分数给模型排名。
